# Meta-Agentic Tree Search · Search Lab 1.17.0

Explore competing workflow designs and export evidence you can independently recompute. This notebook uses the standard-library lab. No model, credentials, paid provider or GPU is required. Inputs are synthetic assumptions; every candidate remains unapproved.

[Open the visual lab](https://montrealai.github.io/AGI-Alpha-Agent-v0/meta_agentic_tree_search_v0/) · [Original notebook](https://github.com/MontrealAI/AGI-Alpha-Agent-v0/blob/main/alpha_factory_v1/demos/meta_agentic_tree_search_v0/research_notebook_archive.ipynb)


In [ ]:
import sys
from pathlib import Path
import subprocess

if not (3, 11) <= sys.version_info[:2] < (3, 14):
    raise RuntimeError("Use Python 3.11–3.13")
import tomllib

VERSION = "1.17.0"
REPOSITORY = next((p for p in [Path.cwd(), *Path.cwd().parents]
                   if (p / "alpha_factory_v1/demos/meta_agentic_tree_search_v0/search_lab.py").is_file()
                   and tomllib.loads((p / "pyproject.toml").read_text())["project"]["version"] == VERSION), None)
if REPOSITORY is None:
    REPOSITORY = Path.cwd() / ("AGI-Alpha-Agent-v" + VERSION)
    if not REPOSITORY.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "v" + VERSION,
                        "https://github.com/MontrealAI/AGI-Alpha-Agent-v0.git", str(REPOSITORY)], check=True)
    declared = tomllib.loads((REPOSITORY / "pyproject.toml").read_text())["project"]["version"]
    if declared != VERSION:
        raise RuntimeError("The existing folder is a different version; use a clean folder")
sys.path.insert(0, str(REPOSITORY.resolve()))
print("Using", REPOSITORY, "version", VERSION)


In [ ]:
from alpha_factory_v1.demos.meta_agentic_tree_search_v0.search_lab import evaluate, read_json, brief, verify, write_bundle
cases = read_json(REPOSITORY / "alpha_factory_v1/demos/meta_agentic_tree_search_v0/scenarios.json")
for case in cases:
    print(case["id"], "—", case["title"])


## Run one search

Choose a case before examining its held-out outcomes. Search freezes the candidate before evaluation. The exhaustive oracle consumes additional compute and never chooses the candidate.


In [ ]:
CASE = "release-design"
scenario = next(case for case in cases if case["id"] == CASE)
report = evaluate(scenario)
print(brief(report))


In [ ]:
result = report["result"]
print("Expanded nodes:", len(result["nodes"]))
print("Root visits:", result["nodes"][0]["visits"])
print("Unique training evaluations:", result["uniqueEvaluations"])
print("Exhaustive training gap:", result["oracle"].get("gapTotal", "not audited"))
for stage, choice in zip(scenario["stages"], result["candidate"]):
    print(stage["label"], "→", stage["choices"][choice]["label"])
print("First iteration:", result["trace"][0])


## Inspect the deliberately unsafe objective

The shortcut case removes the defect penalty from search. Independent held-out defect limits must still block its proposal. This is a model demonstration, not a measurement of deployed reliability.


In [ ]:
for case in cases:
    run = evaluate(case)
    print(case["id"], run["result"]["status"], [(gate["id"], gate["passed"]) for gate in run["result"]["gates"]])


## Export and independently recompute

The six files include the scenario, complete search trace, unapproved proposal, unsubmitted $AGIALPHA review job, review brief and SHA-256 checksums. Existing evidence is never silently overwritten. Download the ZIP from the notebook file browser to retain it outside this session.


In [ ]:
import shutil
OUTPUT = Path("mats-notebook-runs")
folder = write_bundle(report, OUTPUT)
replayed = verify(read_json(folder / "run.json"))
assert replayed == report
archive = shutil.make_archive(str(OUTPUT / ("mats-" + report["sha256"][:12])), "zip", folder)
print("Verified six files:", folder.resolve())
print("Download:", Path(archive).resolve())


## What happens next

Keep the baseline. A real reviewer needs fresh workloads, representative measurements, explicit acceptance criteria and authenticated authority. The lab does not submit jobs, mint assets, transfer funds or deploy the candidate.

[Complete method and preserved research](https://github.com/MontrealAI/AGI-Alpha-Agent-v0/tree/main/alpha_factory_v1/demos/meta_agentic_tree_search_v0) · [Project notice](https://github.com/MontrealAI/AGI-Alpha-Agent-v0/blob/main/docs/DISCLAIMER_SNIPPET.md)
